Desafio -- Etapa 1: extração. Mesma lógica que a professora acabou de
mostrar na Vitrine (scripts/01_extrair_fontes.py), com as MESMAS 3 fontes
(CSV, API, Scraping) -- só que aqui CSV e API vêm da Vitrine em miniatura,
e API/Scraping são fontes públicas DE VERDADE, não mocks.

Fonte 1 -- CSV (análoga ao vendas.csv/produtos.csv da Vitrine, slide 10):
    raw/populacao_2010.csv -- o Censo 2010, já pronto num arquivo. Não
    precisamos buscar isso de novo: é histórico, não muda. Igual à Vitrine
    com os CSVs dela, é só ler.

Fonte 2 -- API (análoga à API de produtos da Vitrine, slide 13):
    GET https://servicodados.ibge.gov.br/api/v1/localidades/estados
    Devolve uma lista de 27 objetos, um por estado + DF, no formato:
        {"id": 35, "sigla": "SP", "nome": "São Paulo",
         "regiao": {"id": 3, "sigla": "SE", "nome": "Sudeste"}}

Fonte 3 -- Web Scraping (análoga ao preço do concorrente, slide 15):
    A página da Wikipédia "Lista de unidades federativas do Brasil por
    população" tem uma tabela (classe "wikitable") com uma linha por estado:
    nome do estado e colunas de população de 2026 (estimativa) e 2022
    (Censo), com um espaço especial (NBSP, "\xa0") como separador de milhar
    em boa parte dos números (ex.: "44\xa0411\xa0238").

    Atenção, pegadinha real: a Wikipédia devolve HTTP 403 (bloqueado) para
    requisições sem um User-Agent próprio -- o requests padrão é tratado
    como robô. Por isso USER_AGENT_WIKIPEDIA abaixo já vem definido; use-o
    em headers=USER_AGENT_WIKIPEDIA no requests.get().

Rode com:  python3 scripts/01_extrair_fontes.py

In [ ]:
# Importação das Bibliotecas utilizadas
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup


In [ ]:
# Mapeamento dados necessários
ROOT = Path.cwd().parent
RAW = ROOT / "raw"

URL_API_ESTADOS = "https://servicodados.ibge.gov.br/api/v1/localidades/estados"
URL_WIKIPEDIA_POPULACAO = "https://pt.wikipedia.org/wiki/Lista_de_unidades_federativas_do_Brasil_por_popula%C3%A7%C3%A3o"
USER_AGENT_WIKIPEDIA = {"User-Agent": "aula-etl-python-fiesc/1.0 (uso educacional)"}


In [33]:
# Extração arquivo "Raw.populacao_2010.csv"

def extrair_populacao_2010_csv() -> pd.DataFrame:
    """Fonte 1: CSV raw/populacao_2010.csv"""
    
    censo2010 = pd.read_csv(RAW / "populacao_2010.csv")
    print(f"Total de linhas: {len(censo2010)}")
    print(censo2010.head())
    
    return censo2010


In [34]:
# Validação do def criado
if __name__ == "__main__":
    print("Executando o script...")
    populacao_2010 = extrair_populacao_2010_csv()
    print("Extração finalizada!")

Executando o script...
Total de linhas: 27
  sigla nome_estado  populacao_2010
0    AC        Acre          733559
1    AL     Alagoas         3120494
2    AM    Amazonas         3483985
3    AP       Amapá          669526
4    BA       Bahia        14016906
Extração finalizada!


In [ ]:
# Extração dados API
def extrair_estados_api() -> pd.DataFrame:
   """Fonte 2: API do IBGE """
   consulta_ibge = requests.get(URL_API_ESTADOS, timeout=30) # Realizando consulta por 30 segundos
   consulta_ibge.raise_for_status() # Valida resposta
   ibge = pd.DataFrame(consulta_ibge.json()) #cria df formato pandas
   ibge['regiao'] = ibge["regiao"].apply(lambda r: r['nome']) 
   ibge = ibge[['id', 'sigla', 'nome', 'regiao']] 
   
   print(f"Total de registros: {len(ibge)}")
   print(ibge.head(3))
   return ibge

In [36]:
# Validação do def criado
if __name__ == "__main__":
    print("Executando o script...")
    ibge = extrair_estados_api()
    print("Extração finalizada!")

Executando o script...
Total de registros: 27
   id sigla      nome regiao
0  11    RO  Rondônia  Norte
1  12    AC      Acre  Norte
2  13    AM  Amazonas  Norte
Extração finalizada!


In [44]:
# Extração Wikipedia
def extrair_populacao_wikipedia() -> pd.DataFrame:
   """Fonte 3: Web Scraping Wikipedia"""  
   
   consulta_wikipedia = requests.get(URL_WIKIPEDIA_POPULACAO, timeout=30, headers=USER_AGENT_WIKIPEDIA)
   consulta_wikipedia.encoding = "utf-8"
   html = BeautifulSoup(consulta_wikipedia.text, "html.parser") # transforma HTML em uma estrutura que podemos navegar e pesquisar
   tabela = html.select_one("table.wikitable") #busca a primeira tabela que possui a classe wikitable
   linhas = tabela.select("tr")[2:]
   dados = []
   
   for tr in linhas:
      celulas = tr.find_all(["th", "td"])
      estado = celulas[0].get_text(strip=True)
      if estado == "Total":
         continue
      pop_2026 = int(celulas[1].get_text(strip=True).replace("\xa0", "").replace(" ", ""))
      pop_2022 = int(celulas[2].get_text(strip=True).replace("\xa0", "").replace(" ", ""))
      dados.append({"estado": estado, "populacao_2022": pop_2022, "populacao_2026": pop_2026})
   
   df = pd.DataFrame(dados)
   print(f"[Wikipedia] {len(df)} estados")
   print(df.head(5))
   return df

In [45]:
# Validação do def criado
if __name__ == "__main__":
    print("Executando o script...")
    df = extrair_populacao_wikipedia()
    print("Extração finalizada!")

Executando o script...
[Wikipedia] 27 estados
           estado  populacao_2022  populacao_2026
0       São Paulo        44411238        46179008
1    Minas Gerais        20539989        21460311
2  Rio de Janeiro        16055174        17225410
3           Bahia        14141626        14889472
4          Paraná        11444380        11952456
Extração finalizada!


In [46]:
if __name__ == "__main__":
    populacao_2010 = extrair_populacao_2010_csv()
    print()
    estados = extrair_estados_api()
    print()
    populacao = extrair_populacao_wikipedia()


Total de linhas: 27
  sigla nome_estado  populacao_2010
0    AC        Acre          733559
1    AL     Alagoas         3120494
2    AM    Amazonas         3483985
3    AP       Amapá          669526
4    BA       Bahia        14016906

Total de registros: 27
   id sigla      nome regiao
0  11    RO  Rondônia  Norte
1  12    AC      Acre  Norte
2  13    AM  Amazonas  Norte

[Wikipedia] 27 estados
           estado  populacao_2022  populacao_2026
0       São Paulo        44411238        46179008
1    Minas Gerais        20539989        21460311
2  Rio de Janeiro        16055174        17225410
3           Bahia        14141626        14889472
4          Paraná        11444380        11952456
